[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/semantica-agi/semantica/blob/main/cookbook/introduction/07_Building_Knowledge_Graphs.ipynb)

# 构建知识图谱

## 概述

本 Notebook 演示如何使用 Semantica 的图构建模块,基于已抽取的实体和关系构建知识图谱。你将学会使用 `GraphBuilder` 和 `EntityResolver`。

**文档**:[API 参考](https://semantica.readthedocs.io/reference/kg/)

### 学习目标

- 抽取实体提及(mention)和关系,并将它们映射为图记录
- 使用 `GraphBuilder` 构建一个图,其边来自实际抽取出的关系
- 使用 `EntityResolver` 合并重复的提及,并重新映射关系的端点
- 使用 `semantica.deduplication` 模块,并给出完整的去重后实体集合

## 安装

从 PyPI 安装 Semantica:

```bash
pip install semantica
# 或安装全部可选依赖:
pip install semantica[all]
```

---

## 步骤 1:抽取实体与关系

从文本中抽取实体提及和关系。示例文本在两个不同的句子里都提到了 `Apple Inc.`,因此我们之后可以展示重复提及是如何被消解为一个规范实体的。

In [ ]:
%pip install semantica

# spaCy models are distributed separately from the spaCy library. This lesson
# relies on the English model to recognize standalone places such as Cupertino.
import sys
import subprocess
import spacy

try:
    spacy.load("en_core_web_sm")
except OSError:
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])


In [ ]:
from semantica.semantic_extract import NERExtractor, RelationExtractor

text = (
    "Apple Inc. is headquartered in Cupertino, California. "
    "Tim Cook is the CEO of Apple Inc. "
    "The company is a technology company."
)

ner_extractor = NERExtractor()
relation_extractor = RelationExtractor()

mentions = ner_extractor.extract(text)
relations = relation_extractor.extract(text, mentions)

print("Entity mentions:")
for mention in mentions:
    print(f"  {mention.text!r:<13} {mention.label:<7} span=[{mention.start_char}:{mention.end_char}]")

print("\nExtracted relations:")
for rel in relations:
    print(f"  {rel.subject.text!r} --{rel.predicate}--> {rel.object.text!r}")

## 步骤 2:构建知识图谱

为每个提及分配一个图 ID,然后把每个关系的 `subject` 和 `object` 转换成这些 ID。根据实际的关系端点来构建边 —— 而不是根据列表位置去猜测端点 —— 才能让图忠实于原文。

In [ ]:
from semantica.kg import GraphBuilder

entities = []
span_to_id = {}
for i, mention in enumerate(mentions, 1):
    graph_id = f"e{i}"
    span_to_id[(mention.start_char, mention.end_char)] = graph_id
    entities.append({
        "id": graph_id,
        "type": mention.label,
        "name": mention.text,
        "properties": {},
    })

relationships = []
for rel in relations:
    source_id = span_to_id.get((rel.subject.start_char, rel.subject.end_char))
    target_id = span_to_id.get((rel.object.start_char, rel.object.end_char))
    if source_id is None or target_id is None:
        print(f"Skipping relation with unmapped endpoint: "
              f"{rel.subject.text!r} --{rel.predicate}--> {rel.object.text!r}")
        continue
    relationships.append({
        "source": source_id,
        "target": target_id,
        "type": rel.predicate,
        "properties": {},
    })

builder = GraphBuilder()
knowledge_graph = builder.build({"entities": entities, "relationships": relationships})

id_to_name = {entity["id"]: entity["name"] for entity in entities}

print(f"Graph entities ({len(knowledge_graph['entities'])}):")
for entity in knowledge_graph["entities"]:
    print(f"  {entity['id']}: {entity['name']} ({entity['type']})")

print(f"\nGraph relationships ({len(knowledge_graph['relationships'])}):")
for relationship in knowledge_graph["relationships"]:
    print(f"  {id_to_name[relationship['source']]} "
          f"--{relationship['type']}--> {id_to_name[relationship['target']]}")

edges = {
    (id_to_name[r["source"]], r["type"], id_to_name[r["target"]])
    for r in knowledge_graph["relationships"]
}
assert ("Apple Inc.", "located_in", "Cupertino") in edges
assert ("Tim Cook", "works_for", "Apple Inc.") in edges

## 步骤 3:实体消解

图中目前为同一个组织包含了两个节点。`EntityResolver` 会把重复的提及合并为一个规范实体,并记录被合并的源 ID(`merged_from`),这样关系端点就可以被重新映射到规范实体上。

In [ ]:
from semantica.kg import EntityResolver

entity_resolver = EntityResolver()
resolved_entities = entity_resolver.resolve_entities(entities)

canonical_id = {}
for entity in resolved_entities:
    for source_id in entity.get("merged_from", [entity["id"]]):
        canonical_id[source_id] = entity["id"]
    if entity.get("merged_from"):
        print(f"Merged {entity['merged_from']} -> {entity['id']}: {entity['name']}")

print(f"\nMentions in: {len(entities)}, resolved entities out: {len(resolved_entities)}")

resolved_names = {entity["id"]: entity["name"] for entity in resolved_entities}
print("\nRelationships remapped onto canonical entities:")
for relationship in relationships:
    source = canonical_id[relationship["source"]]
    target = canonical_id[relationship["target"]]
    print(f"  {resolved_names[source]} --{relationship['type']}--> {resolved_names[target]}")

canonical_entities = {(entity["name"], entity["type"]) for entity in resolved_entities}
assert canonical_entities == {
    ("Apple Inc.", "ORG"),
    ("Tim Cook", "PERSON"),
    ("Cupertino", "GPE"),
    ("California", "GPE"),
}
assert len(resolved_entities) == 4

## 步骤 4:去重

`semantica.deduplication` 模块让你对同一问题拥有更精细的控制。注意,`merge_duplicates` 为每个重复*组*返回一个 `MergeOperation` —— 完整的去重后集合,是这些合并后的实体,加上不属于任何组的所有实体。

In [ ]:
from semantica.deduplication import DuplicateDetector, EntityMerger, MergeStrategy

detector = DuplicateDetector(similarity_threshold=0.8)
duplicate_groups = detector.detect_duplicate_groups(entities)
print(f"Duplicate groups: {len(duplicate_groups)}")
for group in duplicate_groups:
    print(f"  {[entity['name'] for entity in group.entities]} "
          f"(confidence={group.confidence:.2f})")

merger = EntityMerger()
merge_operations = merger.merge_duplicates(
    entities, strategy=MergeStrategy.KEEP_MOST_COMPLETE
)

merged_source_ids = {
    entity["id"] for op in merge_operations for entity in op.source_entities
}
untouched_entities = [e for e in entities if e["id"] not in merged_source_ids]
deduplicated_entities = untouched_entities + [
    op.merged_entity for op in merge_operations
]

print(f"\nMerge operations: {len(merge_operations)}")
print(f"Deduplicated entities ({len(deduplicated_entities)}):")
for entity in deduplicated_entities:
    print(f"  {entity['id']}: {entity['name']} ({entity['type']})")

assert len(merge_operations) == 1
assert len(deduplicated_entities) == 4

## 总结

你已经学会如何构建知识图谱:

- **从抽取到图**:将每个提及映射为一个图 ID,并根据实际的 `Relation.subject` / `Relation.object` 端点构建边
- **GraphBuilder**:根据显式的 `{"entities": ..., "relationships": ...}` 输入构建知识图谱
- **EntityResolver**:将重复的提及合并为规范实体,并重新映射关系端点
- **去重**:把 `MergeOperation` 的结果与未改动的实体合并,得到完整的去重集合

下一步:在 Graph_Analytics Notebook 中学习如何分析图。